# Anthony Burgess RAG Project - Q&A App using LangChain, OpenAI and Pinecone

### Requirements and Environment Setup

In [1]:
pip install -q -r ./requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# .env files are not included in the repository for security reasons (as specified
# in the .gitignore file), so create your own .env file with the following content:
# OPENAI_API_KEY=""
# PINECONE_API_KEY=""
# LANGCHAIN_API_KEY=""
# LANGCHAIN_TRACING_V2="false"

True

### Loading Documents

In [3]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    if extension == '.pdf':
        from langchain_classic.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain_classic.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain_classic.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data

### Chunking Data

In [4]:
def chunk_data(data, chunk_size=256):
    from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

### Calculating Cost

In [5]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

### Embedding and Uploading to the Pinecone Vector Database

In [6]:
# - give every chunk a deterministic Pinecone ID based on
# - its document source, page, page label and chunk text
# - check each chunk individually
# - skip chunks for which ID already exists in Pinecone
# - only insert new chunks

def insert_or_fetch_embeddings(index_name, chunks, namespace):
    """Insert only new embeddings into Pinecone, then return the vector store."""
    import hashlib
    import time
    import pinecone
    from langchain_community.vectorstores import Pinecone as PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    def make_chunk_id(chunk):
        source = str(chunk.metadata.get("source", "unknown-source")).replace("\\", "/")
        page = str(chunk.metadata.get("page", "unknown-page"))
        page_label = str(chunk.metadata.get("page_label", "unknown-page-label"))
        id_text = f"{source}|{page}|{page_label}|{chunk.page_content}"
        return hashlib.sha256(id_text.encode("utf-8")).hexdigest()

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"Creating index {index_name} ... ", end="")

        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)

        print("Ok")

    pinecone_index = pc.Index(index_name)

    vector_store = PineconeVectorStore(
        index=pinecone_index,
        embedding=embeddings,
        text_key="text",
        namespace=namespace
    )

    chunk_ids = [make_chunk_id(chunk) for chunk in chunks]
    chunks_to_add = []
    ids_to_add = []

    fetch_batch_size = 100

    for i in range(0, len(chunk_ids), fetch_batch_size):
        batch_ids = chunk_ids[i:i + fetch_batch_size]
        batch_chunks = chunks[i:i + fetch_batch_size]

        fetch_response = pinecone_index.fetch(
            ids=batch_ids,
            namespace=namespace
        )

        existing_vectors = getattr(fetch_response, "vectors", None)

        if existing_vectors is None:
            existing_vectors = fetch_response.get("vectors", {})

        existing_ids = set(existing_vectors.keys())

        for chunk_id, chunk in zip(batch_ids, batch_chunks):
            if chunk_id not in existing_ids:
                ids_to_add.append(chunk_id)
                chunks_to_add.append(chunk)

    if chunks_to_add:
        print(f"Adding {len(chunks_to_add)} new documents to index {index_name} ...\n", end="")

        embedding_batch_size = 100
        total_batches = (len(chunks_to_add) + embedding_batch_size - 1) // embedding_batch_size

        for i in range(0, len(chunks_to_add), embedding_batch_size):
            batch_chunks = chunks_to_add[i:i + embedding_batch_size]
            batch_ids = ids_to_add[i:i + embedding_batch_size]
            batch_texts = [chunk.page_content for chunk in batch_chunks]

            batch_number = (i // embedding_batch_size) + 1
            print(f"Adding batch {batch_number}/{total_batches} ({len(batch_chunks)} documents) ... ", end="")

            batch_embeddings = embeddings.embed_documents(batch_texts)

            vectors = []

            for chunk_id, chunk, embedding in zip(batch_ids, batch_chunks, batch_embeddings):
                metadata = dict(chunk.metadata)
                metadata["text"] = chunk.page_content

                vectors.append({
                    "id": chunk_id,
                    "values": embedding,
                    "metadata": metadata
                })

            pinecone_index.upsert(
                vectors=vectors,
                namespace=namespace
            )

            print("Ok")

        print("All batches added successfully!")

        # Give Pinecone a moment to make freshly upserted vectors visible to fetch/stats.
        time.sleep(5)
    else:
        print(f"All {len(chunks)} documents already exist in index {index_name}. No new documents added.")

    return vector_store

In [7]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('OK')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('OK')

### Asking and Getting Answers

In [8]:
def ask_and_get_answer(vector_store, q, k=3):
    from langchain_classic.chains import RetrievalQA
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=1)

    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    chain = RetrievalQA.from_chain_type(llm=llm,
                                        chain_type="stuff",
                                        retriever=retriever,
                                        return_source_documents=True)
    
    answer = chain.invoke(q)
    return answer

### Running Code

#### Load Documents and Chunk Data

In [10]:
from pathlib import Path

# amend accordingly
folder_paths = [
    Path("../Anthony_Burgess_Docs"),
]

data = []

for folder_path in folder_paths:
    for file_path in folder_path.iterdir():
        if file_path.suffix.lower() in [".pdf", ".docx", ".txt"]:
            document_pages = load_document(str(file_path))

            if document_pages is not None:
                for page in document_pages:
                    page.metadata["page_label"] = page.metadata.get("page_label", "Unknown page")
                    page.metadata["page"] = page.metadata.get("page", "Unknown page")

                data.extend(document_pages)

print(f"There are {len(data)} pages in total")

if data:
    print(data[0].page_content)
    print(data[0].metadata)
    print(f"There are {len(data[0].page_content)} characters on the first page")

Loading ..\Anthony_Burgess_Docs\00 Nothing Like the Sun Norton 2013.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 13 MAY 1978 Let's Have a Bloody Revolution.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 13 MAY 1978.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 15 APR 1978 Outlooks.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 15 APR 1978.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 15 SEP 1979 Funereal in Berlin.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 2 FEB 1980 Passport Please.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 2 SEP 1978 Quite a Piece of Ameringlish Hopefully.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 20 JAN 1979 The Art of the Unconscious.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 21 JUL 1979 Let Them Eat Red.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 22 JUL 1978.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 22 JULY 1978 Germans and Other Absurdities.pdf
Loading ..\Anthony_Burgess_Docs\Blue Coast 23 JUN 1979 Prime Time for Hitler.pdf
Loading ..\

In [11]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[len(chunks) - 1].page_content)

6381
036BB_tx.indd   310 25/05/2012   15:38


In [12]:
print_embedding_cost(chunks)

Total Tokens: 398631
Embedding Cost in USD: 0.007973


In [16]:
import pinecone
pc = pinecone.Pinecone()
pc.list_indexes()

IndexList([<name='anthony-burgess-rag', dim=1536, ready=True>, <name='ou-comp-it-degree', dim=1536, ready=True>])

In [14]:
from pinecone import ServerlessSpec
index_name = 'anthony-burgess-rag'
if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name} ...', end='')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print('Index created!')
else:
    print(f'Index {index_name} already exists!')
# pc.list_indexes()[0]['name']
pc.describe_index('anthony-burgess-rag')

Creating index: anthony-burgess-rag ...Index created!


IndexModel(name='anthony-burgess-rag', metric='cosine', host='https://anthony-burgess-rag-moalxso.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [18]:
index_name = 'anthony-burgess-rag'
index = pc.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=6381, metric='cosine', namespaces=1)

In [15]:
# Only use this if you need to
# delete_pinecone_index('anthony-burgess-rag')

In [19]:
index_name = 'anthony-burgess-rag'
namespace = 'anthony-burgess-rag'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks, namespace=namespace)

All 6381 documents already exist in index anthony-burgess-rag. No new documents added.


In [20]:
q = 'Write four paragraphs about P.R. Deltoid. Answer from the provided context only.'
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])
print("\nReference(s):")

for doc in answer["source_documents"]:
    source = doc.metadata.get("source", "Unknown document")
    page_label = doc.metadata.get("page_label")
    page = doc.metadata.get("page")

    print(f"\nDocument source: {source}")
    print(f"Page label: {page_label}")
    print(f"Page index: {page}")

P.R. Deltoid is introduced as the Post-Corrective Adviser in the context provided. His presence is characterized by a certain gravitas, as he comes into the scene with an air of authority. The mention of his office being in the same building suggests he is someone who plays a significant role in overseeing the actions of those he advises, particularly in relation to the protagonist, Alex. His tired appearance hints at the weight of his responsibilities and the challenges he faces in dealing with troubled youths.

During a particular interaction, P.R. Deltoid's concern for Alex becomes evident when he acknowledges the trouble that has arisen. His inquiry about whether "it’s happened" suggests that he is aware of Alex's previous patterns of behavior and the potential consequences that have come to fruition. This creates a dynamic where Deltoid is not only a figure of authority but also someone who bears witness to the struggles of those under his guidance.

The presence of the "millicent

#### While Loop for Asking Questions

In [23]:
import time
i = 1
print('Enter your questions (or write Quit to quit).')

while True:
    q = input(f'Question #{i}: ') + ' Answer from the provided context only.'
    i = i + 1
    if q.lower().startswith('quit'):
        print('\nQuitting ... bye bye!')
        time.sleep(2)
        break
    
    answer = ask_and_get_answer(vector_store, q)
    print(f'\n{"-" * 50}')
    print(f'\nQuery: ' + answer['query'])
    print(f'\nAnswer: ' + answer['result'])
    print("\nReference(s):")

    for doc in answer["source_documents"]:
        source = doc.metadata.get("source", "Unknown document")
        page_label = doc.metadata.get("page_label")
        page = doc.metadata.get("page")

        print(f"\nDocument: {source}")
        print(f"Page label: {page_label}")
        print(f"Page index: {page}")


Enter your questions (or write Quit to quit).

--------------------------------------------------

Query: Where are Crepes Suzettes and Peaches Melba consumed? Answer from the provided context only.

Answer: Crepes suzette and peches Melba are consumed in the Hotel de Paris.

Reference(s):

Document: ..\Anthony_Burgess_Docs\Blue Coast 15 APR 1978.pdf
Page label: 1
Page index: 0

Document: ..\Anthony_Burgess_Docs\Blue Coast 15 APR 1978 Outlooks.pdf
Page label: 1
Page index: 0

Document: ..\Anthony_Burgess_Docs\Blue Coast 21 JUL 1979 Let Them Eat Red.pdf
Page label: 1
Page index: 0

--------------------------------------------------

Query: What is the form BF35/792/N044dj? Answer from the provided context only.

Answer: The context does not provide specific details about what the form BF35/792/N044dj is. It only mentions that it is required to enter the United States and that one must apply to the embassy or consulate to obtain it.

Reference(s):

Document: ..\Anthony_Burgess_Docs\Blue 